In [ ]:
import os

os.environ["HF_TOKEN"] = "hf_"

from convokitai import Corpus
from convokit.redirection.config import DEFAULT_LORA_CONFIG, DEFAULT_TRAIN_CONFIG
from convokit.redirection.gemmaLikelihoodModel import GemmaLikelihoodModel
from convokitai_evaluation import MediatorRedirection

corpus = Corpus.load("sample_corpus")

In [2]:
# off-the-shelf model, no fine-tuning
likelihood_model = GemmaLikelihoodModel(
    hf_token=os.getenv("HF_TOKEN"),
    model_id="google/gemma-2b",
    device="mps",  # "cuda" | "mps" | "cpu"
    train_config=dict(DEFAULT_TRAIN_CONFIG, max_seq_length=2048),
    bnb_config=None,  # 4-bit quantization needs CUDA
    lora_config=DEFAULT_LORA_CONFIG,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
# counterfactual "prev": the mediator's previous turn stands in for the one being scored
redirection = MediatorRedirection(likelihood_model, counterfactual="prev")
redirection.transform(corpus)

Computing contexts
5 / 12
10 / 12
Computing actual likelihoods
5 / 2
10 / 2
Computing reference likelihoods
5 / 2
10 / 2
Computing redirection scores
5 / 12
10 / 12


In [4]:
# one score per mediator turn that a participant replied to
convo = next(c for c in corpus.iter_conversations() if c.meta["mediator"] == "constructive")
for utt in convo.iter_utterances():
    if utt.meta.get("redirection") is not None:
        print(f"{utt.meta['redirection']:+.2f}", utt.text[:90])

-1.67 Welcome to the discussion, Goose and Bear.
-0.25 It is wonderful to see you both here.
-0.63 It is a delight to be here with you both.
+0.75 I am so happy to be joining you both today.


In [5]:
# mean per conversation; then per mediator type
summary = redirection.summarize(corpus)
summary["mediator"] = summary.conversation_id.map(lambda c: corpus.get_conversation(c).meta["mediator"])
summary.groupby("mediator").redirection_mean.mean()

mediator
constructive    5.328749
greeting       -0.066763
paraphrase      0.631521
Name: redirection_mean, dtype: float64

In [6]:
summary

,conversation_id,n_scored,redirection_mean,mediator
0,mtj4ur0f-efa1m0,4,-0.448919,greeting
1,mtj4ursf-dmdhx8,4,-0.590827,greeting
2,mtj4us6f-2xlxgt,4,0.556515,greeting
3,mtj4ushn-a6u955,4,0.216178,greeting
4,mtj4urnt-axjca7,4,-2.175371,paraphrase
5,mtj4us0y-aajy2k,4,0.407640,paraphrase
6,mtj4ush6-a3a03d,4,2.024950,paraphrase
7,mtj4usq7-ifjk38,2,2.268867,paraphrase
8,muiqt1a3-h7bdu0,4,9.306146,constructive
9,muiqt1dk-ziab7u,4,-0.158389,constructive
